1. **PREPROCESSING**

In [2]:
import pandas as pd
import numpy as np

# Load data
klaim = pd.read_csv('dataset/Data_Klaim.csv')
polis = pd.read_csv('dataset/Data_Polis.csv')
# Merge
data = pd.merge(klaim, polis, on='Nomor Polis')

# =============================
# Feature Engineering
# =============================

# Convert tanggal lahir ke umur
data['Tanggal Lahir'] = pd.to_datetime(data['Tanggal Lahir'], format='%Y%m%d')
today = pd.to_datetime('today')

data['umur'] = (today - data['Tanggal Lahir']).dt.days // 365

# Buat jumlah klaim per nasabah
jumlah_klaim = data.groupby('Nomor Polis').size().reset_index(name='jumlah_klaim')
data = pd.merge(data, jumlah_klaim, on='Nomor Polis')

# =============================
# Target (Risiko Klaim)
# =============================

# Kita buat rule awal (semi-pakar)
def label_risiko(row):
    if row['Nominal Biaya RS Yang Terjadi'] > 100000000:
        return 1  # Risiko tinggi
    elif row['jumlah_klaim'] > 2:
        return 1
    else:
        return 0  # Risiko rendah

data['risiko'] = data.apply(label_risiko, axis=1)

# =============================
# Encoding
# =============================

data = pd.get_dummies(data, columns=['Gender', 'Plan Code', 'Domisili', 'Lokasi RS'], drop_first=True)

# =============================
# Final Dataset
# =============================

X = data[['umur', 'jumlah_klaim', 'Nominal Biaya RS Yang Terjadi']]
y = data['risiko']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X.head())

   umur  jumlah_klaim  Nominal Biaya RS Yang Terjadi
0    59            18                   6.143948e+06
1    69            22                   8.230952e+07
2    66            25                   1.928599e+08
3    66            25                   1.914244e+08
4    57            14                   1.389364e+08


**2.Forward Chaining**

In [3]:
def forward_chaining(row):
    if row['Nominal Biaya RS Yang Terjadi'] > 100000000:
        return 1

    if row['jumlah_klaim'] > 2:
        return 1

    if row['umur'] > 60:
        return 1

    return 0

data['pred_forward'] = data.apply(forward_chaining, axis=1)

Sistem membaca data nasabah
Lalu mencocokkan dengan rule (aturan pakar)
Jika kondisi terpenuhi akan langsung hasilkan keputusan

3. **BACKWARD CHAINING**

In [4]:
def backward_chaining(row):
    # Hipotesis: Risiko Tinggi (1)

    if row['Nominal Biaya RS Yang Terjadi'] > 100000000:
        return 1

    if row['jumlah_klaim'] > 2:
        return 1

    if row['umur'] > 60:
        return 1

    return 0

data['pred_backward'] = data.apply(backward_chaining, axis=1)

Backward chaining bekerja dengan memulai dari tujuan atau hipotesis, lalu memeriksa apakah data mendukung hipotesis tersebut

Apakah nasabah ini berisiko tinggi?
Lalu sistem cek:
apakah biaya besar?
apakah klaim sering?

Kalau iya berarti benar

**4. CERTAINTY FACTOR (CF)**

In [5]:
def certainty_factor(row):
    cf = 0

    if row['Nominal Biaya RS Yang Terjadi'] > 100000000:
        cf += 0.8

    if row['jumlah_klaim'] > 2:
        cf += 0.7

    if row['umur'] > 60:
        cf += 0.6

    cf = min(cf, 1)

    if cf > 0.7:
        return 1, cf
    else:
        return 0, cf

cf_result = data.apply(certainty_factor, axis=1)

data['pred_cf'] = [x[0] for x in cf_result]
data['nilai_cf'] = [x[1] for x in cf_result]

Certainty Factor digunakan untuk menangani ketidakpastian dengan memberikan nilai keyakinan pada setiap aturan, sehingga hasilnya tidak hanya keputusan, tetapi juga tingkat kepercayaan.

Setiap kondisi punya nilai keyakinan
Contoh:
umur tua → 0.6
klaim banyak → 0.7
biaya besar → 0.8

Lalu dijumlahkan jadi tingkat kepercayaan

**5. NAIVE BAYES**

In [6]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

model = GaussianNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Akurasi:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Akurasi: 0.6781857451403888
              precision    recall  f1-score   support

           0       0.31      0.69      0.43       161
           1       0.91      0.68      0.78       765

    accuracy                           0.68       926
   macro avg       0.61      0.68      0.60       926
weighted avg       0.81      0.68      0.72       926



Naive Bayes adalah metode berbasis probabilitas yang menghitung kemungkinan suatu data termasuk dalam kelas tertentu berdasarkan pola dari data sebelumnya.

**6. PERBANDINGAN METODE**

In [7]:
from sklearn.metrics import accuracy_score

# Samakan index test
data_test = data.iloc[X_test.index]

acc_forward = accuracy_score(y_test, data_test['pred_forward'])
acc_backward = accuracy_score(y_test, data_test['pred_backward'])
acc_cf = accuracy_score(y_test, data_test['pred_cf'])

print("Forward:", acc_forward)
print("Backward:", acc_backward)
print("Certainty Factor:", acc_cf)

Forward: 0.9222462203023758
Backward: 0.9222462203023758
Certainty Factor: 0.652267818574514


Forward Chaining & Backward Chaining (92.22%) → memiliki akurasi yang sama karena menggunakan rule yang identik, perbedaannya hanya pada cara penalaran (forward: dari data ke keputusan, backward: dari tujuan ke data), sehingga hasilnya tetap konsisten.
Akurasi tinggi (92%) → menunjukkan bahwa rule yang dibuat sudah sesuai dengan pola pada dataset, sehingga sistem pakar bekerja dengan baik.
Certainty Factor (65.11%) → memiliki akurasi lebih rendah karena menggunakan bobot keyakinan yang bersifat subjektif dan sangat bergantung pada nilai CF serta threshold yang ditentukan.
CF tidak berbasis data langsung → sehingga kurang mampu menangkap pola sebenarnya dibanding metode rule langsung.
Kesimpulan → metode rule-based (Forward & Backward) lebih akurat pada kasus ini, sedangkan Certainty Factor lebih fleksibel tetapi kurang presisi karena bergantung pada asumsi pakar.